# Project Phase 3: Modeling (Vector-leaf GBDT)

このノートブックでは、CatBoostの MultiRegression を使用し、5日後の全会合期待金利を一括予測します。

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.modeling import prepare_modeling_data, train_vector_leaf_model

%matplotlib inline
plt.rcParams['font.family'] = 'AppleGothic'

## 1. データのロードと目視確認
学習に使用する $X$ (特徴量) と $Y$ (ターゲット) の中身を確認します。

In [ ]:
df_featured = pd.read_csv('../data/featured_data.csv')
df_featured['日付'] = pd.to_datetime(df_featured['日付'])

X, Y, dates, feature_cols, target_cols = prepare_modeling_data(df_featured, target_horizon=5)

print("=== 入力特徴量 (X) のプレビュー ===")
display(X.head())

print("\n=== 予測対象 (Y: 5日後のプレミアム) のプレビュー ===")
display(Y.head())

print(f"\n学習期間: {dates.iloc[0].strftime('%Y-%m-%d')} 〜 {dates.iloc[-1].strftime('%Y-%m-%d')}")
print(f"サンプル数: {len(X)}")

## 2. モデルの学習 (Vector-leaf GBDT)

In [ ]:
model, X_test, Y_test = train_vector_leaf_model(X, Y)
predictions = model.predict(X_test)
df_pred = pd.DataFrame(predictions, columns=target_cols, index=Y_test.index)

## 3. 予測結果と政策金利の可視化

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 14))
test_dates = dates.loc[Y_test.index]

axes[0].step(test_dates, X_test['Actual_Policy_Rate'], color='black', lw=3, label='Actual Policy Rate', alpha=0.8, where='post')
axes[0].plot(test_dates, Y_test['M1_target_5d'], label='Actual M1 Spread (t+5)', alpha=0.5, color='blue')
axes[0].plot(test_dates, df_pred['M1_target_5d'], label='Predicted M1 Spread', color='red', lw=2)
axes[0].set_title('M1 5-day Prediction: Spread vs Policy Rate', fontsize=14)
axes[0].legend(loc='upper left')

target_day_idx = -1
actual_curve = [0.0] + Y_test.iloc[target_day_idx].tolist()
pred_curve = [0.0] + df_pred.iloc[target_day_idx].tolist()
axes[1].plot(range(9), actual_curve, marker='o', label='Actual Spread Path (t+5)')
axes[1].plot(range(9), pred_curve, marker='o', color='red', label='Predicted Spread Path')
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.3)
axes[1].set_title(f'Expected Path Comparison: {test_dates.iloc[target_day_idx].strftime("%Y-%m-%d")}')
axes[1].set_xticks(range(9))
axes[1].set_xticklabels(['Actual'] + [f'M{i}' for i in range(1, 9)])
axes[1].legend()
plt.tight_layout()
plt.show()

## 4. 特徴量重要度

In [ ]:
importances = model.get_feature_importance()
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10, 8))
sns.barplot(x=importances[indices], y=[feature_cols[i] for i in indices])
plt.title('Feature Importance (Ranked)')
plt.show()